In [3]:

# NOTEBOOK 03 — Load preprocessed artifacts.
# No model training.

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

artifact_path = Path("artifacts") / "nexusiq_preprocessed.joblib"

if not artifact_path.exists():
    raise FileNotFoundError(
        f"Cannot find {artifact_path.resolve()}. "
        "Run the artifact-saving cell in Notebook 02 first, "
        "and check that both notebooks use the same project folder."
    )

artifacts = joblib.load(artifact_path)

X_train_champion = artifacts["X_train_champion"]
X_train_champion_spells = artifacts["X_train_champion_spells"]
y_train = artifacts["y_train"]
audit_train = artifacts["audit_train"]

X_test_champion = artifacts["X_test_champion"]
X_test_champion_spells = artifacts["X_test_champion_spells"]
y_test = artifacts["y_test"]
audit_test = artifacts["audit_test"]

print("Loaded artifact:", artifact_path.resolve())
print("Train champion-only:", X_train_champion.shape)
print("Train champion + spells:", X_train_champion_spells.shape)
print("Test champion-only:", X_test_champion.shape)
print("Test champion + spells:", X_test_champion_spells.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

assert X_train_champion.shape == (118675, 272)
assert X_train_champion_spells.shape == (118675, 290)
assert X_test_champion.shape == (42399, 272)
assert X_test_champion_spells.shape == (42399, 290)

print("\nArtifacts loaded and validated. No models trained.")


Loaded artifact: C:\Users\Administrator\Self Study\NexusIQ\notebooks\artifacts\nexusiq_preprocessed.joblib
Train champion-only: (118675, 272)
Train champion + spells: (118675, 290)
Test champion-only: (42399, 272)
Test champion + spells: (42399, 290)
Training target: (118675,)
Test target: (42399,)

Artifacts loaded and validated. No models trained.


In [4]:

# NOTEBOOK 03 — Chronological validation split
# No model training or fitting.

train_dates = pd.to_numeric(audit_train["creation"], errors="raise")
validation_cutoff = train_dates.quantile(0.80)

train_mask = train_dates < validation_cutoff
val_mask = ~train_mask

# Champion-only features
X_train_champ_fit = X_train_champion.loc[train_mask]
X_val_champ = X_train_champion.loc[val_mask]

# Champion + spell features
X_train_spells_fit = X_train_champion_spells.loc[train_mask]
X_val_spells = X_train_champion_spells.loc[val_mask]

# Shared targets and audit metadata
y_train_fit = y_train.loc[train_mask]
y_val = y_train.loc[val_mask]

audit_train_fit = audit_train.loc[train_mask]
audit_val = audit_train.loc[val_mask]

print("CHRONOLOGICAL VALIDATION SPLIT")
print("Training matches:", len(y_train_fit))
print("Validation matches:", len(y_val))
print("Final test matches (untouched):", len(y_test))

print("\nDATE BOUNDARIES")
print("Training latest:", audit_train_fit["creation"].max())
print("Validation earliest:", audit_val["creation"].min())
print("Validation latest:", audit_val["creation"].max())

print("\nBLUE-WIN RATES")
print("Training:", round(float(y_train_fit.mean()), 4))
print("Validation:", round(float(y_val.mean()), 4))
print("Test:", round(float(y_test.mean()), 4))

print("\nVALIDATION CHECKS")
print("Training/validation IDs disjoint:",
      set(y_train_fit.index).isdisjoint(set(y_val.index)))
print("Training strictly earlier than validation:",
      audit_train_fit["creation"].max() < audit_val["creation"].min())
print("Champion columns aligned:",
      X_train_champ_fit.columns.equals(X_val_champ.columns))
print("Spell columns aligned:",
      X_train_spells_fit.columns.equals(X_val_spells.columns))

assert len(y_train_fit) + len(y_val) == len(y_train)
assert set(y_train_fit.index).isdisjoint(set(y_val.index))
assert audit_train_fit["creation"].max() < audit_val["creation"].min()
assert X_train_champ_fit.columns.equals(X_val_champ.columns)
assert X_train_spells_fit.columns.equals(X_val_spells.columns)

print("\nValidation split prepared. No models trained.")


CHRONOLOGICAL VALIDATION SPLIT
Training matches: 94940
Validation matches: 23735
Final test matches (untouched): 42399

DATE BOUNDARIES
Training latest: 1494685720999
Validation earliest: 1494685723669
Validation latest: 1494993866205

BLUE-WIN RATES
Training: 0.5125
Validation: 0.5154
Test: 0.519

VALIDATION CHECKS
Training/validation IDs disjoint: True
Training strictly earlier than validation: True
Champion columns aligned: True
Spell columns aligned: True

Validation split prepared. No models trained.


In [6]:

from pathlib import Path
import sklearn
import joblib
import os

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("CPU cores:", os.cpu_count())
print("\nExisting artifacts:")

for path in sorted(artifact_dir.iterdir()):
    if path.is_file():
        print(f"{path.name} — {path.stat().st_size / 1024**2:.2f} MB")
    elif path.is_dir():
        print(f"{path.name}/")

print("\nRequired data available:")
for name in [
    "X_train_champ_fit", "X_val_champ",
    "X_train_spells_fit", "X_val_spells",
    "y_train_fit", "y_val", "X_test_champion",
    "X_test_champion_spells", "y_test",
]:
    print(f"{name}: {name in globals()}")

print("\nTraining and validation sizes:")
if "y_train_fit" in globals() and "y_val" in globals():
    print("Training:", len(y_train_fit))
    print("Validation:", len(y_val))
    print("Final test:", len(y_test))


scikit-learn: 1.9.1
joblib: 1.5.3
CPU cores: 16

Existing artifacts:
nexusiq_preprocessed.joblib — 7.58 MB

Required data available:
X_train_champ_fit: True
X_val_champ: True
X_train_spells_fit: True
X_val_spells: True
y_train_fit: True
y_val: True
X_test_champion: True
X_test_champion_spells: True
y_test: True

Training and validation sizes:
Training: 94940
Validation: 23735
Final test: 42399


In [7]:

import numpy as np
import pandas as pd
from pathlib import Path

# Reuse existing variables; never rebuild the dataset here.
required = [
    "X_train_champ_fit", "X_train_spells_fit",
    "y_train_fit", "audit_train_fit",
    "X_val_champ", "X_val_spells", "y_val",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables: {missing}")

# Sort all training-period data by timestamp, keeping rows aligned.
timestamps = pd.to_numeric(
    audit_train_fit["creation"], errors="raise"
).to_numpy()

order = np.argsort(timestamps, kind="stable")

X_champ_ordered = X_train_champ_fit.iloc[order].reset_index(drop=True)
X_spells_ordered = X_train_spells_fit.iloc[order].reset_index(drop=True)
y_ordered = y_train_fit.iloc[order].reset_index(drop=True)
time_ordered = timestamps[order]

# Expanding-window folds: earlier observations train each fold,
# and the immediately following time window validates it.
fold_boundaries = [
    (0.50, 0.65),
    (0.65, 0.80),
    (0.80, 1.00),
]

cv_splits = []
for train_end_q, val_end_q in fold_boundaries:
    train_end_time = np.quantile(time_ordered, train_end_q)
    val_end_time = np.quantile(time_ordered, val_end_q)

    train_idx = np.flatnonzero(time_ordered < train_end_time)
    val_idx = np.flatnonzero(
        (time_ordered >= train_end_time) &
        (time_ordered <= val_end_time)
    )

    if len(train_idx) == 0 or len(val_idx) == 0:
        raise ValueError("A chronological fold is empty.")

    assert time_ordered[train_idx].max() < time_ordered[val_idx].min()

    cv_splits.append((train_idx, val_idx))

    print(
        f"Fold {len(cv_splits)} | "
        f"train={len(train_idx):,} | validation={len(val_idx):,} | "
        f"train win rate={y_ordered.iloc[train_idx].mean():.4f} | "
        f"validation win rate={y_ordered.iloc[val_idx].mean():.4f}"
    )

print("\nOrdered feature shapes:")
print("Champion-only:", X_champ_ordered.shape)
print("Champion + spells:", X_spells_ordered.shape)
print("Target:", y_ordered.shape)

print("\nFinal validation and test remain separate:")
print("Validation:", len(y_val))
print("Final test:", len(y_test))
print("\nCV preparation complete. No models trained.")


Fold 1 | train=47,470 | validation=14,241 | train win rate=0.5096 | validation win rate=0.5160
Fold 2 | train=61,711 | validation=14,241 | train win rate=0.5111 | validation win rate=0.5128
Fold 3 | train=75,952 | validation=18,988 | train win rate=0.5114 | validation win rate=0.5167

Ordered feature shapes:
Champion-only: (94940, 272)
Champion + spells: (94940, 290)
Target: (94940,)

Final validation and test remain separate:
Validation: 23735
Final test: 42399

CV preparation complete. No models trained.


In [8]:

from pathlib import Path
import json
import joblib
import numpy as np

from scipy.stats import loguniform, randint, uniform
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

# Reusable output directories
MODEL_DIR = Path("artifacts") / "models"
SEARCH_DIR = Path("artifacts") / "search_results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_ITER_SEARCH = 20
CV_FOLDS = cv_splits
N_JOBS = 2  # Keep RAM use controlled

# Pipelines ensure each fold learns preprocessing from its own training data.
model_configs = {
    "logistic_regression": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )),
        ]),
        "params": {
            "model__C": loguniform(1e-4, 1e3),
            "model__solver": ["lbfgs"],
            "model__class_weight": [None, "balanced"],
        },
    },

    "random_forest": {
        "pipeline": Pipeline([
            ("model", RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=1
            )),
        ]),
        "params": {
            "model__n_estimators": randint(200, 801),
            "model__max_depth": [None, 8, 12, 16, 24, 32],
            "model__min_samples_split": randint(2, 31),
            "model__min_samples_leaf": randint(1, 16),
            "model__max_features": ["sqrt", 0.4, 0.6, 0.8],
            "model__class_weight": [None, "balanced_subsample"],
        },
    },

    "hist_gradient_boosting": {
        "pipeline": Pipeline([
            ("model", HistGradientBoostingClassifier(
                random_state=RANDOM_STATE,
                early_stopping=False
            )),
        ]),
        "params": {
            "model__learning_rate": loguniform(0.02, 0.3),
            "model__max_iter": randint(100, 501),
            "model__max_leaf_nodes": [7, 15, 31, 63],
            "model__max_depth": [None, 5, 8, 12],
            "model__min_samples_leaf": randint(10, 101),
            "model__l2_regularization": loguniform(1e-4, 10),
        },
    },
}

feature_sets = {
    "champion_only": (X_champ_ordered, X_val_champ),
    "champion_spells": (X_spells_ordered, X_val_spells),
}

# A stable description of the planned search, for audit purposes.
search_plan = {
    "random_state": RANDOM_STATE,
    "n_iter": N_ITER_SEARCH,
    "cv_folds": len(CV_FOLDS),
    "scoring": ["average_precision", "roc_auc", "accuracy", "neg_log_loss"],
    "refit_metric": "average_precision",
    "feature_sets": list(feature_sets),
    "models": list(model_configs),
    "note": "Chronological CV; final test excluded from tuning.",
}

with open(SEARCH_DIR / "search_plan.json", "w", encoding="utf-8") as f:
    json.dump(search_plan, f, indent=2)

print("Pipelines defined:", list(model_configs))
print("Feature sets:", list(feature_sets))
print("Searches planned:", len(model_configs) * len(feature_sets))
print("Candidates per search:", N_ITER_SEARCH)
print("CV folds per candidate:", len(CV_FOLDS))
print("Maximum candidate-fold fits:", 
      len(model_configs) * len(feature_sets) * N_ITER_SEARCH * len(CV_FOLDS))
print("Parallel jobs:", N_JOBS)
print("Search cache:", SEARCH_DIR.resolve())
print("\nConfiguration saved. No models trained.")


Pipelines defined: ['logistic_regression', 'random_forest', 'hist_gradient_boosting']
Feature sets: ['champion_only', 'champion_spells']
Searches planned: 6
Candidates per search: 20
CV folds per candidate: 3
Maximum candidate-fold fits: 360
Parallel jobs: 2
Search cache: C:\Users\Administrator\Self Study\NexusIQ\notebooks\artifacts\search_results

Configuration saved. No models trained.


In [9]:

import json
import joblib
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV

search_name = "logistic_regression__champion_only"
result_path = SEARCH_DIR / f"{search_name}.joblib"

if result_path.exists():
    print("Loading previously completed search...")
    search = joblib.load(result_path)
    print("Loaded:", result_path.name)
else:
    print("Tuning Logistic Regression — champion-only features.")
    print("This will fit models; it may take a few minutes.")

    config = model_configs["logistic_regression"]

    search = RandomizedSearchCV(
        estimator=config["pipeline"],
        param_distributions=config["params"],
        n_iter=N_ITER_SEARCH,
        scoring={
            "average_precision": "average_precision",
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "neg_log_loss": "neg_log_loss",
        },
        refit="average_precision",
        cv=CV_FOLDS,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1,
        return_train_score=False,
        error_score="raise",
    )

    search.fit(X_champ_ordered, y_ordered)

    # Save only after the search completes successfully.
    temporary_path = result_path.with_suffix(".tmp")
    joblib.dump(search, temporary_path, compress=3)
    temporary_path.replace(result_path)
    print("Search saved:", result_path)

results = pd.DataFrame(search.cv_results_)
best_idx = search.best_index_

print("\nBest cross-validation average precision:",
      round(search.best_score_, 5))
print("Best parameters:")
print(json.dumps(search.best_params_, indent=2, default=str))

print("\nBest configuration's mean CV metrics:")
for metric in ["average_precision", "roc_auc", "accuracy", "neg_log_loss"]:
    value = results.loc[best_idx, f"mean_test_{metric}"]
    if metric == "neg_log_loss":
        value = -value
        print(f"Log loss: {value:.5f}")
    else:
        print(f"{metric}: {value:.5f}")

print("\nTop 5 configurations by average precision:")
display(
    results.sort_values("rank_test_average_precision")[
        [
            "rank_test_average_precision",
            "mean_test_average_precision",
            "std_test_average_precision",
            "mean_test_roc_auc",
            "mean_test_accuracy",
            "mean_test_neg_log_loss",
            "params",
        ]
    ].head(5)
)


Tuning Logistic Regression — champion-only features.
This will fit models; it may take a few minutes.
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Search saved: artifacts\search_results\logistic_regression__champion_only.joblib

Best cross-validation average precision: 0.56043
Best parameters:
{
  "model__C": 0.0009472334467618556,
  "model__class_weight": "balanced",
  "model__solver": "lbfgs"
}

Best configuration's mean CV metrics:
average_precision: 0.56043
roc_auc: 0.55196
accuracy: 0.53379
Log loss: 0.68907

Top 5 configurations by average precision:


,rank_test_average_precision,mean_test_average_precision,std_test_average_precision,mean_test_roc_auc,mean_test_accuracy,mean_test_neg_log_loss,params
14,1,0.560432,0.005109,0.551965,0.533793,-0.689067,"{'model__C': 0.0009472334467618556, 'model__cl..."
10,2,0.560424,0.004950,0.551990,0.534525,-0.689212,"{'model__C': 0.0019223460470643655, 'model__cl..."
1,3,0.560423,0.004950,0.551990,0.534525,-0.689212,"{'model__C': 0.0019232844888462217, 'model__cl..."
4,4,0.560383,0.005291,0.551919,0.533629,-0.688706,"{'model__C': 0.00025502648504032837, 'model__c..."
7,5,0.560379,0.005291,0.551919,0.533665,-0.688700,"{'model__C': 0.0002482453105132191, 'model__cl..."


In [10]:

search_name = "logistic_regression__champion_spells"
result_path = SEARCH_DIR / f"{search_name}.joblib"

if result_path.exists():
    print("Loading previously completed search...")
    search = joblib.load(result_path)
    print("Loaded:", result_path.name)
else:
    print("Tuning Logistic Regression — champion + spells.")
    config = model_configs["logistic_regression"]

    search = RandomizedSearchCV(
        estimator=config["pipeline"],
        param_distributions=config["params"],
        n_iter=N_ITER_SEARCH,
        scoring={
            "average_precision": "average_precision",
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "neg_log_loss": "neg_log_loss",
        },
        refit="average_precision",
        cv=CV_FOLDS,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1,
        return_train_score=False,
        error_score="raise",
    )

    search.fit(X_spells_ordered, y_ordered)

    temporary_path = result_path.with_suffix(".tmp")
    joblib.dump(search, temporary_path, compress=3)
    temporary_path.replace(result_path)
    print("Search saved:", result_path)

results = pd.DataFrame(search.cv_results_)
best_idx = search.best_index_

print("\nBest CV average precision:", round(search.best_score_, 5))
print("Best parameters:")
print(json.dumps(search.best_params_, indent=2, default=str))

print("\nBest configuration's CV metrics:")
for metric in ["average_precision", "roc_auc", "accuracy", "neg_log_loss"]:
    value = results.loc[best_idx, f"mean_test_{metric}"]
    if metric == "neg_log_loss":
        print(f"Log loss: {-value:.5f}")
    else:
        print(f"{metric}: {value:.5f}")

print("\nTop 5 configurations:")
display(
    results.sort_values("rank_test_average_precision")[
        [
            "rank_test_average_precision",
            "mean_test_average_precision",
            "std_test_average_precision",
            "mean_test_roc_auc",
            "mean_test_accuracy",
            "mean_test_neg_log_loss",
            "params",
        ]
    ].head(5)
)


Tuning Logistic Regression — champion + spells.
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Search saved: artifacts\search_results\logistic_regression__champion_spells.joblib

Best CV average precision: 0.56232
Best parameters:
{
  "model__C": 0.0009472334467618556,
  "model__class_weight": "balanced",
  "model__solver": "lbfgs"
}

Best configuration's CV metrics:
average_precision: 0.56232
roc_auc: 0.55427
accuracy: 0.53636
Log loss: 0.68871

Top 5 configurations:


,rank_test_average_precision,mean_test_average_precision,std_test_average_precision,mean_test_roc_auc,mean_test_accuracy,mean_test_neg_log_loss,params
14,1,0.562317,0.004949,0.554273,0.536356,-0.688715,"{'model__C': 0.0009472334467618556, 'model__cl..."
10,2,0.562296,0.004849,0.554265,0.536555,-0.688869,"{'model__C': 0.0019223460470643655, 'model__cl..."
1,3,0.562295,0.004849,0.554265,0.536555,-0.688869,"{'model__C': 0.0019232844888462217, 'model__cl..."
3,4,0.562258,0.004696,0.554228,0.536959,-0.688676,"{'model__C': 0.13207763079904547, 'model__clas..."
0,5,0.562257,0.004701,0.554229,0.536865,-0.688667,"{'model__C': 0.041858227295469716, 'model__cla..."


In [12]:

from pathlib import Path

print("Saved search results:")
for path in sorted(SEARCH_DIR.glob("*")):
    if path.is_file():
        print(f"{path.name} — {path.stat().st_size / 1024**2:.2f} MB")

rf_path = SEARCH_DIR / "random_forest__champion_only.joblib"
print("\nCompleted Random Forest search saved:", rf_path.exists())

print("\nCompleted Logistic Regression searches:")
for name in [
    "logistic_regression__champion_only.joblib",
    "logistic_regression__champion_spells.joblib",
]:
    print(f"{name}: {(SEARCH_DIR / name).exists()}")


Saved search results:
logistic_regression__champion_only.joblib — 0.36 MB
logistic_regression__champion_spells.joblib — 0.36 MB
search_plan.json — 0.00 MB

Completed Random Forest search saved: False

Completed Logistic Regression searches:
logistic_regression__champion_only.joblib: True
logistic_regression__champion_spells.joblib: True


In [13]:

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from scipy.stats import randint
import joblib
import pandas as pd
import json

search_name = "random_forest__champion_only__lite"
result_path = SEARCH_DIR / f"{search_name}.joblib"

# Use two of the chronological folds to reduce runtime.
lite_cv = CV_FOLDS[:2]

if result_path.exists():
    print("Loading saved lightweight Random Forest search...")
    search = joblib.load(result_path)
else:
    lite_pipeline = Pipeline([
        ("model", RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ])

    lite_params = {
        "model__n_estimators": [80, 120, 200],
        "model__max_depth": [8, 14, 20],
        "model__min_samples_split": [2, 10],
        "model__min_samples_leaf": [2, 5, 10],
        "model__max_features": ["sqrt", 0.5],
        "model__class_weight": [None],
    }

    search = RandomizedSearchCV(
        estimator=lite_pipeline,
        param_distributions=lite_params,
        n_iter=8,
        scoring={
            "average_precision": "average_precision",
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "neg_log_loss": "neg_log_loss",
        },
        refit="average_precision",
        cv=lite_cv,
        random_state=RANDOM_STATE,
        n_jobs=2,
        verbose=1,
        return_train_score=False,
        error_score="raise",
    )

    search.fit(X_champ_ordered, y_ordered)

    temp_path = result_path.with_suffix(".tmp")
    joblib.dump(search, temp_path, compress=3)
    temp_path.replace(result_path)
    print("Search saved:", result_path)

results = pd.DataFrame(search.cv_results_)
best_idx = search.best_index_

print("\nBest CV average precision:", round(search.best_score_, 5))
print("Best parameters:")
print(json.dumps(search.best_params_, indent=2, default=str))

print("\nBest configuration's CV metrics:")
for metric in ["average_precision", "roc_auc", "accuracy", "neg_log_loss"]:
    value = results.loc[best_idx, f"mean_test_{metric}"]
    print(
        f"{'Log loss' if metric == 'neg_log_loss' else metric}: "
        f"{-value if metric == 'neg_log_loss' else value:.5f}"
    )

display(
    results.sort_values("rank_test_average_precision")[
        [
            "rank_test_average_precision",
            "mean_test_average_precision",
            "std_test_average_precision",
            "mean_test_roc_auc",
            "mean_test_accuracy",
            "mean_test_neg_log_loss",
            "params",
        ]
    ].head(5)
)


Fitting 2 folds for each of 8 candidates, totalling 16 fits
Search saved: artifacts\search_results\random_forest__champion_only__lite.joblib

Best CV average precision: 0.5547
Best parameters:
{
  "model__n_estimators": 120,
  "model__min_samples_split": 10,
  "model__min_samples_leaf": 5,
  "model__max_features": "sqrt",
  "model__max_depth": 8,
  "model__class_weight": null
}

Best configuration's CV metrics:
average_precision: 0.55470
roc_auc: 0.54382
accuracy: 0.52897
Log loss: 0.69090


,rank_test_average_precision,mean_test_average_precision,std_test_average_precision,mean_test_roc_auc,mean_test_accuracy,mean_test_neg_log_loss,params
1,1,0.554698,0.003411,0.543818,0.528966,-0.690895,"{'model__n_estimators': 120, 'model__min_sampl..."
2,2,0.554103,0.004577,0.544756,0.528966,-0.690880,"{'model__n_estimators': 120, 'model__min_sampl..."
3,3,0.554020,0.003847,0.544905,0.533109,-0.689753,"{'model__n_estimators': 200, 'model__min_sampl..."
7,4,0.553515,0.001917,0.543876,0.531494,-0.690234,"{'model__n_estimators': 80, 'model__min_sample..."
0,5,0.552420,0.002604,0.543697,0.533776,-0.689804,"{'model__n_estimators': 200, 'model__min_sampl..."


In [14]:

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform, randint
import joblib
import pandas as pd
import json

search_name = "hist_gradient_boosting__champion_only__lite"
result_path = SEARCH_DIR / f"{search_name}.joblib"

if result_path.exists():
    print("Loading saved HistGradientBoosting search...")
    search = joblib.load(result_path)
else:
    pipeline = Pipeline([
        ("model", HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
            early_stopping=False
        ))
    ])

    params = {
        "model__learning_rate": loguniform(0.03, 0.2),
        "model__max_iter": randint(80, 251),
        "model__max_leaf_nodes": [7, 15, 31],
        "model__max_depth": [None, 5, 8],
        "model__min_samples_leaf": randint(20, 81),
        "model__l2_regularization": loguniform(0.001, 5),
    }

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=10,
        scoring={
            "average_precision": "average_precision",
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "neg_log_loss": "neg_log_loss",
        },
        refit="average_precision",
        cv=CV_FOLDS[:2],
        random_state=RANDOM_STATE,
        n_jobs=2,
        verbose=1,
        return_train_score=False,
        error_score="raise",
    )

    search.fit(X_champ_ordered, y_ordered)

    temp_path = result_path.with_suffix(".tmp")
    joblib.dump(search, temp_path, compress=3)
    temp_path.replace(result_path)
    print("Search saved:", result_path)

results = pd.DataFrame(search.cv_results_)
best_idx = search.best_index_

print("\nBest CV average precision:", round(search.best_score_, 5))
print("Best parameters:")
print(json.dumps(search.best_params_, indent=2, default=str))

print("\nBest configuration's CV metrics:")
for metric in ["average_precision", "roc_auc", "accuracy", "neg_log_loss"]:
    value = results.loc[best_idx, f"mean_test_{metric}"]
    print(
        f"{'Log loss' if metric == 'neg_log_loss' else metric}: "
        f"{-value if metric == 'neg_log_loss' else value:.5f}"
    )

display(
    results.sort_values("rank_test_average_precision")[
        [
            "rank_test_average_precision",
            "mean_test_average_precision",
            "std_test_average_precision",
            "mean_test_roc_auc",
            "mean_test_accuracy",
            "mean_test_neg_log_loss",
            "params",
        ]
    ].head(5)
)


Fitting 2 folds for each of 10 candidates, totalling 20 fits
Search saved: artifacts\search_results\hist_gradient_boosting__champion_only__lite.joblib

Best CV average precision: 0.55529
Best parameters:
{
  "model__l2_regularization": 0.32842573382490514,
  "model__learning_rate": 0.0705166262783807,
  "model__max_depth": 5,
  "model__max_iter": 211,
  "model__max_leaf_nodes": 7,
  "model__min_samples_leaf": 79
}

Best configuration's CV metrics:
average_precision: 0.55529
roc_auc: 0.54693
accuracy: 0.53381
Log loss: 0.68911


,rank_test_average_precision,mean_test_average_precision,std_test_average_precision,mean_test_roc_auc,mean_test_accuracy,mean_test_neg_log_loss,params
6,1,0.555286,0.002967,0.546928,0.533811,-0.689112,{'model__l2_regularization': 0.328425733824905...
5,2,0.553899,0.002976,0.545976,0.533144,-0.689238,{'model__l2_regularization': 0.193816746768763...
1,3,0.553869,0.004221,0.545565,0.533214,-0.689506,{'model__l2_regularization': 0.003776663327107...
9,4,0.552766,0.004265,0.544273,0.533530,-0.689409,{'model__l2_regularization': 2.309664459647273...
7,5,0.550608,0.001728,0.542505,0.531845,-0.689756,{'model__l2_regularization': 0.121223333270458...


In [15]:

search_name = "hist_gradient_boosting__champion_spells__lite"
result_path = SEARCH_DIR / f"{search_name}.joblib"

if result_path.exists():
    print("Loading saved search...")
    search = joblib.load(result_path)
else:
    pipeline = Pipeline([
        ("model", HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
            early_stopping=False
        ))
    ])

    params = {
        "model__learning_rate": loguniform(0.03, 0.2),
        "model__max_iter": randint(80, 251),
        "model__max_leaf_nodes": [7, 15, 31],
        "model__max_depth": [None, 5, 8],
        "model__min_samples_leaf": randint(20, 81),
        "model__l2_regularization": loguniform(0.001, 5),
    }

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=10,
        scoring={
            "average_precision": "average_precision",
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "neg_log_loss": "neg_log_loss",
        },
        refit="average_precision",
        cv=CV_FOLDS[:2],
        random_state=RANDOM_STATE,
        n_jobs=2,
        verbose=1,
        return_train_score=False,
        error_score="raise",
    )

    search.fit(X_spells_ordered, y_ordered)

    temp_path = result_path.with_suffix(".tmp")
    joblib.dump(search, temp_path, compress=3)
    temp_path.replace(result_path)
    print("Search saved:", result_path)

results = pd.DataFrame(search.cv_results_)
best_idx = search.best_index_

print("\nBest CV average precision:", round(search.best_score_, 5))
print("Best parameters:")
print(json.dumps(search.best_params_, indent=2, default=str))

print("\nBest configuration's CV metrics:")
for metric in ["average_precision", "roc_auc", "accuracy", "neg_log_loss"]:
    value = results.loc[best_idx, f"mean_test_{metric}"]
    print(
        f"{'Log loss' if metric == 'neg_log_loss' else metric}: "
        f"{-value if metric == 'neg_log_loss' else value:.5f}"
    )

display(
    results.sort_values("rank_test_average_precision")[
        [
            "rank_test_average_precision",
            "mean_test_average_precision",
            "std_test_average_precision",
            "mean_test_roc_auc",
            "mean_test_accuracy",
            "mean_test_neg_log_loss",
            "params",
        ]
    ].head(5)
)


Fitting 2 folds for each of 10 candidates, totalling 20 fits
Search saved: artifacts\search_results\hist_gradient_boosting__champion_spells__lite.joblib

Best CV average precision: 0.55945
Best parameters:
{
  "model__l2_regularization": 0.003776663327107336,
  "model__learning_rate": 0.0403316977206044,
  "model__max_depth": 8,
  "model__max_iter": 167,
  "model__max_leaf_nodes": 7,
  "model__min_samples_leaf": 55
}

Best configuration's CV metrics:
average_precision: 0.55945
roc_auc: 0.54984
accuracy: 0.53476
Log loss: 0.68888


,rank_test_average_precision,mean_test_average_precision,std_test_average_precision,mean_test_roc_auc,mean_test_accuracy,mean_test_neg_log_loss,params
1,1,0.559449,0.000731,0.549840,0.534759,-0.688882,{'model__l2_regularization': 0.003776663327107...
6,2,0.558944,0.001160,0.549874,0.535040,-0.688559,{'model__l2_regularization': 0.328425733824905...
5,3,0.558086,0.000581,0.549143,0.535812,-0.688691,{'model__l2_regularization': 0.193816746768763...
2,4,0.556933,0.000927,0.548277,0.534443,-0.688833,{'model__l2_regularization': 0.003376448065950...
9,5,0.556912,0.000459,0.549122,0.535988,-0.688704,{'model__l2_regularization': 2.309664459647273...


In [16]:

from pathlib import Path
import joblib
import pandas as pd

search_files = [
    ("Logistic Regression", "Champion-only",
     "logistic_regression__champion_only.joblib", 3),
    ("Logistic Regression", "Champion + spells",
     "logistic_regression__champion_spells.joblib", 3),
    ("Random Forest", "Champion-only",
     "random_forest__champion_only__lite.joblib", 2),
    ("HistGradientBoosting", "Champion-only",
     "hist_gradient_boosting__champion_only__lite.joblib", 2),
    ("HistGradientBoosting", "Champion + spells",
     "hist_gradient_boosting__champion_spells__lite.joblib", 2),
]

rows = []
for model_name, features, filename, folds in search_files:
    path = SEARCH_DIR / filename

    if not path.exists():
        print("Missing:", filename)
        continue

    fitted_search = joblib.load(path)
    idx = fitted_search.best_index_
    cv_results = fitted_search.cv_results_

    rows.append({
        "Model": model_name,
        "Features": features,
        "CV folds": folds,
        "Candidates": len(cv_results["params"]),
        "Average precision": cv_results["mean_test_average_precision"][idx],
        "AP std": cv_results["std_test_average_precision"][idx],
        "ROC-AUC": cv_results["mean_test_roc_auc"][idx],
        "Accuracy": cv_results["mean_test_accuracy"][idx],
        "Log loss": -cv_results["mean_test_neg_log_loss"][idx],
        "Best parameters": fitted_search.best_params_,
        "Search file": filename,
    })

comparison = pd.DataFrame(rows).sort_values(
    "Average precision", ascending=False
).reset_index(drop=True)

display(
    comparison[
        [
            "Model", "Features", "CV folds", "Candidates",
            "Average precision", "AP std", "ROC-AUC",
            "Accuracy", "Log loss",
        ]
    ].round(5)
)

print("\nTop candidate's parameters:")
print(comparison.iloc[0]["Best parameters"])

comparison_path = SEARCH_DIR / "model_comparison.csv"
comparison.to_csv(comparison_path, index=False)
print("\nComparison saved:", comparison_path)


,Model,Features,CV folds,Candidates,Average precision,AP std,ROC-AUC,Accuracy,Log loss
0,Logistic Regression,Champion + spells,3,20,0.56232,0.00495,0.55427,0.53636,0.68871
1,Logistic Regression,Champion-only,3,20,0.56043,0.00511,0.55196,0.53379,0.68907
2,HistGradientBoosting,Champion + spells,2,10,0.55945,0.00073,0.54984,0.53476,0.68888
3,HistGradientBoosting,Champion-only,2,10,0.55529,0.00297,0.54693,0.53381,0.68911
4,Random Forest,Champion-only,2,8,0.55470,0.00341,0.54382,0.52897,0.69090



Top candidate's parameters:
{'model__C': np.float64(0.0009472334467618556), 'model__class_weight': 'balanced', 'model__solver': 'lbfgs'}

Comparison saved: artifacts\search_results\model_comparison.csv


In [17]:
import joblib
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    log_loss,
    brier_score_loss,
)

finalists = [
    (
        "Logistic Regression",
        "Champion + spells",
        "logistic_regression__champion_spells.joblib",
        X_val_spells,
    ),
    (
        "Logistic Regression",
        "Champion-only",
        "logistic_regression__champion_only.joblib",
        X_val_champ,
    ),
    (
        "HistGradientBoosting",
        "Champion + spells",
        "hist_gradient_boosting__champion_spells__lite.joblib",
        X_val_spells,
    ),
]

rows = []

for model_name, features, filename, X_val in finalists:
    search = joblib.load(SEARCH_DIR / filename)
    model = search.best_estimator_

    probabilities = model.predict_proba(X_val)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    rows.append({
        "Model": model_name,
        "Features": features,
        "Validation AP": average_precision_score(y_val, probabilities),
        "Validation ROC-AUC": roc_auc_score(y_val, probabilities),
        "Validation Accuracy": accuracy_score(y_val, predictions),
        "Validation Log Loss": log_loss(y_val, probabilities),
        "Validation Brier": brier_score_loss(y_val, probabilities),
    })

validation_comparison = (
    pd.DataFrame(rows)
    .sort_values("Validation AP", ascending=False)
    .reset_index(drop=True)
)

display(validation_comparison.round(5))

output_path = SEARCH_DIR / "validation_comparison.csv"
validation_comparison.to_csv(output_path, index=False)
print("Saved:", output_path)

,Model,Features,Validation AP,Validation ROC-AUC,Validation Accuracy,Validation Log Loss,Validation Brier
0,Logistic Regression,Champion + spells,0.56957,0.56103,0.54363,0.68704,0.24697
1,Logistic Regression,Champion-only,0.56756,0.55851,0.54013,0.68746,0.24718
2,HistGradientBoosting,Champion + spells,0.56319,0.55437,0.53937,0.68819,0.24754


Saved: artifacts\search_results\validation_comparison.csv


In [18]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
)

search = joblib.load(
    SEARCH_DIR / "logistic_regression__champion_spells.joblib"
)
final_model = search.best_estimator_

val_probabilities = final_model.predict_proba(X_val_spells)[:, 1]

threshold_rows = []

for threshold in np.arange(0.35, 0.651, 0.025):
    predictions = (val_probabilities >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": round(float(threshold), 3),
        "Accuracy": accuracy_score(y_val, predictions),
        "Balanced accuracy": balanced_accuracy_score(y_val, predictions),
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1": f1_score(y_val, predictions, zero_division=0),
        "Predicted Blue win rate": predictions.mean(),
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(4))

threshold_path = SEARCH_DIR / "validation_thresholds.csv"
threshold_table.to_csv(threshold_path, index=False)
print("Saved:", threshold_path)

,Threshold,Accuracy,Balanced accuracy,Precision,Recall,F1,Predicted Blue win rate
0,0.350,0.5178,0.5027,0.5168,0.9945,0.6801,0.9920
1,0.375,0.5217,0.5069,0.5190,0.9867,0.6802,0.9800
2,0.400,0.5253,0.5113,0.5214,0.9627,0.6764,0.9518
3,0.425,0.5328,0.5206,0.5269,0.9161,0.6690,0.8962
4,0.450,0.5373,0.5280,0.5329,0.8298,0.6490,0.8027
5,0.475,0.5435,0.5383,0.5439,0.7091,0.6156,0.6720
6,0.500,0.5436,0.5432,0.5573,0.5574,0.5573,0.5155
7,0.525,0.5341,0.5386,0.5696,0.3938,0.4657,0.3564
8,0.550,0.5246,0.5333,0.5913,0.2514,0.3528,0.2192
9,0.575,0.5107,0.5225,0.6100,0.1408,0.2287,0.1189


Saved: artifacts\search_results\validation_thresholds.csv


In [19]:
import json
import joblib
from pathlib import Path

final_model_path = MODEL_DIR / "nexusiq_logistic_champion_spells.joblib"
metadata_path = MODEL_DIR / "nexusiq_model_metadata.json"

joblib.dump(final_model, final_model_path)

metadata = {
    "model": "Logistic Regression",
    "features": "champion + team spell counts",
    "threshold": 0.50,
    "target": "blue_win",
    "dataset_period": "2016-2017",
    "validation_average_precision": 0.56957,
    "validation_roc_auc": 0.56103,
    "validation_accuracy_at_0_5": 0.54363,
    "validation_balanced_accuracy_at_0_5": 0.5436,
    "note": (
        "Historical baseline; not validated for current patches. "
        "Threshold selected provisionally using validation data."
    ),
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Model saved:", final_model_path)
print("Metadata saved:", metadata_path)
print("Model type:", type(final_model).__name__)
print("Threshold:", metadata["threshold"])

Model saved: artifacts\models\nexusiq_logistic_champion_spells.joblib
Metadata saved: artifacts\models\nexusiq_model_metadata.json
Model type: Pipeline
Threshold: 0.5


In [20]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
    classification_report,
)

final_model = joblib.load(
    MODEL_DIR / "nexusiq_logistic_champion_spells.joblib"
)

with open(
    MODEL_DIR / "nexusiq_model_metadata.json",
    encoding="utf-8",
) as f:
    metadata = json.load(f)

threshold = metadata["threshold"]
test_probabilities = final_model.predict_proba(X_test_champion_spells)[:, 1]
test_predictions = (test_probabilities >= threshold).astype(int)

test_metrics = {
    "Average precision": average_precision_score(y_test, test_probabilities),
    "ROC-AUC": roc_auc_score(y_test, test_probabilities),
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Balanced accuracy": balanced_accuracy_score(y_test, test_predictions),
    "Precision": precision_score(y_test, test_predictions, zero_division=0),
    "Recall": recall_score(y_test, test_predictions, zero_division=0),
    "F1": f1_score(y_test, test_predictions, zero_division=0),
    "Log loss": log_loss(y_test, test_probabilities),
    "Brier score": brier_score_loss(y_test, test_probabilities),
}

test_results = pd.DataFrame(
    [{"Metric": k, "Test value": v} for k, v in test_metrics.items()]
)
display(test_results.round(5))

print("Test matches:", len(y_test))
print("Actual Blue win rate:", round(float(y_test.mean()), 5))
print("Predicted Blue win rate:", round(float(test_predictions.mean()), 5))
print("\nConfusion matrix (rows=actual, columns=predicted):")
print(confusion_matrix(y_test, test_predictions))
print("\nClassification report:")
print(classification_report(y_test, test_predictions, zero_division=0))

results_path = SEARCH_DIR / "final_test_metrics.csv"
test_results.to_csv(results_path, index=False)
print("\nSaved:", results_path)

,Metric,Test value
0,Average precision,0.56572
1,ROC-AUC,0.55558
2,Accuracy,0.53945
3,Balanced accuracy,0.53868
4,Precision,0.55599
5,Recall,0.55894
6,F1,0.55746
7,Log loss,0.68843
8,Brier score,0.24764


Test matches: 42399
Actual Blue win rate: 0.51897
Predicted Blue win rate: 0.52173

Confusion matrix (rows=actual, columns=predicted):
[[10573  9822]
 [ 9705 12299]]

Classification report:
              precision    recall  f1-score   support

           0       0.52      0.52      0.52     20395
           1       0.56      0.56      0.56     22004

    accuracy                           0.54     42399
   macro avg       0.54      0.54      0.54     42399
weighted avg       0.54      0.54      0.54     42399


Saved: artifacts\search_results\final_test_metrics.csv


In [21]:
import joblib
from pathlib import Path

model_path = Path("artifacts/models/nexusiq_logistic_champion_spells.joblib")
model = joblib.load(model_path)

feature_names = list(model.feature_names_in_)

print("Model:", type(model).__name__)
print("Expected features:", len(feature_names))
print("\nFirst 20 features:")
for name in feature_names[:20]:
    print(" -", name)

print("\nLast 30 features:")
for name in feature_names[-30:]:
    print(" -", name)

print("\nPipeline steps:")
for name, step in model.steps:
    print(f" - {name}: {type(step).__name__}")

Model: Pipeline
Expected features: 290

First 20 features:
 - blue_champ_1
 - blue_champ_2
 - blue_champ_3
 - blue_champ_4
 - blue_champ_5
 - blue_champ_6
 - blue_champ_7
 - blue_champ_8
 - blue_champ_9
 - blue_champ_10
 - blue_champ_11
 - blue_champ_12
 - blue_champ_13
 - blue_champ_14
 - blue_champ_15
 - blue_champ_16
 - blue_champ_17
 - blue_champ_18
 - blue_champ_19
 - blue_champ_20

Last 30 features:
 - red_champ_254
 - red_champ_266
 - red_champ_267
 - red_champ_268
 - red_champ_412
 - red_champ_420
 - red_champ_421
 - red_champ_427
 - red_champ_429
 - red_champ_432
 - red_champ_497
 - red_champ_498
 - blue_spell_1
 - blue_spell_3
 - blue_spell_4
 - blue_spell_6
 - blue_spell_7
 - blue_spell_11
 - blue_spell_12
 - blue_spell_14
 - blue_spell_21
 - red_spell_1
 - red_spell_3
 - red_spell_4
 - red_spell_6
 - red_spell_7
 - red_spell_11
 - red_spell_12
 - red_spell_14
 - red_spell_21

Pipeline steps:
 - scaler: StandardScaler
 - model: LogisticRegression


In [22]:
from pathlib import Path
import joblib

project_root = Path.cwd().parent
artifact_path = project_root / "notebooks" / "artifacts" / "nexusiq_preprocessed.joblib"

print("PROJECT FILES")
for path in sorted(project_root.iterdir()):
    if path.name not in {".git", "__pycache__", ".venv", "venv"}:
        print(f"{'[DIR]' if path.is_dir() else '[FILE]'} {path.name}")

print("\nPREPROCESSING ARTIFACT")
print("Path:", artifact_path)
print("Exists:", artifact_path.exists())

if artifact_path.exists():
    artifact = joblib.load(artifact_path)
    print("Artifact type:", type(artifact).__name__)

    if isinstance(artifact, dict):
        print("\nArtifact keys:")
        for key, value in artifact.items():
            if hasattr(value, "shape"):
                print(f" - {key}: {type(value).__name__}, shape={value.shape}")
            elif isinstance(value, (list, tuple, dict, set)):
                print(f" - {key}: {type(value).__name__}, length={len(value)}")
            else:
                print(f" - {key}: {type(value).__name__}")

PROJECT FILES
[DIR] data
[DIR] notebooks
[DIR] src

PREPROCESSING ARTIFACT
Path: c:\Users\Administrator\Self Study\NexusIQ\notebooks\artifacts\nexusiq_preprocessed.joblib
Exists: True
Artifact type: dict

Artifact keys:
 - X_train_champion: DataFrame, shape=(118675, 272)
 - X_train_champion_spells: DataFrame, shape=(118675, 290)
 - y_train: Series, shape=(118675,)
 - audit_train: DataFrame, shape=(118675, 2)
 - X_test_champion: DataFrame, shape=(42399, 272)
 - X_test_champion_spells: DataFrame, shape=(42399, 290)
 - y_test: Series, shape=(42399,)
 - audit_test: DataFrame, shape=(42399, 2)


In [23]:
from pathlib import Path

project_root = Path.cwd().parent

for folder_name in ["src", "notebooks", "data"]:
    folder = project_root / folder_name
    print(f"\n--- {folder_name}/ ---")

    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue
        if any(part in {".ipynb_checkpoints", "__pycache__", ".venv", "venv"}
               for part in path.parts):
            continue

        # Show paths only, not dataset contents.
        print(path.relative_to(project_root))


--- src/ ---

--- notebooks/ ---
notebooks\01_eda.ipynb
notebooks\02_preprocessing.ipynb
notebooks\03_modeling.ipynb
notebooks\artifacts\models\nexusiq_logistic_champion_spells.joblib
notebooks\artifacts\models\nexusiq_model_metadata.json
notebooks\artifacts\nexusiq_preprocessed.joblib
notebooks\artifacts\search_results\final_test_metrics.csv
notebooks\artifacts\search_results\hist_gradient_boosting__champion_only__lite.joblib
notebooks\artifacts\search_results\hist_gradient_boosting__champion_spells__lite.joblib
notebooks\artifacts\search_results\logistic_regression__champion_only.joblib
notebooks\artifacts\search_results\logistic_regression__champion_spells.joblib
notebooks\artifacts\search_results\model_comparison.csv
notebooks\artifacts\search_results\random_forest__champion_only__lite.joblib
notebooks\artifacts\search_results\search_plan.json
notebooks\artifacts\search_results\validation_comparison.csv
notebooks\artifacts\search_results\validation_thresholds.csv

--- data/ ---
da